# EDA: YMAD Cars (`mad.csv`)

Цель: провести быстрый, но практичный EDA по метаданным датасета `YMAD` и выявить проблемы качества данных перед обучением.

Что анализируем:
- размер и структуру данных;
- пропуски и дубликаты;
- распределения `brand` / `model` / `color`;
- покрытие `view_id` и `car_id`;
- целостность `url` относительно `car_id/view_id`.

> Примечание: в каталоге есть только CSV с URL-ами, сами изображения не скачаны локально.

In [27]:
!pip install polars opencv-python aiohttp

In [21]:
from pathlib import Path
import csv
import re
from collections import Counter, defaultdict
from statistics import mean, median
import math

import polars as pl

import asyncio
import aiohttp
from tqdm.asyncio import tqdm as atqdm

import cv2
import numpy as np

In [22]:
def check_creation(path: Path) -> str:
    if path.exists():
        return f"Каталог/файл {path} создан!"
    return f"Ошибка в создании каталога/файла {path}!"

In [23]:
HF_DATA_URI = 'hf://datasets/yandex/mad-cars/mad.csv'
DATA_DIR = Path('root/carcv/')
DATA_DIR.mkdir(parents=True, exist_ok=True)
DATA_PATH = Path(DATA_DIR/'mad.csv')

In [24]:
CSV_SCHEMA = {
    'color': pl.Utf8,
    'model': pl.Utf8,
    'brand': pl.Utf8,
    'car_id': pl.Utf8,
    'view_id': pl.Utf8,
    'url': pl.Utf8,
}

In [7]:
def read_mad_csv(path: str | Path) -> pl.DataFrame:
    return pl.read_csv(path, schema_overrides=CSV_SCHEMA)

if DATA_PATH.exists():
    df = read_mad_csv(DATA_PATH)
else:
    print('Локальный mad.csv не найден, загружаю из Hugging Face через Polars...')
    try:
        # Для приватного доступа: huggingface-cli login
        df = read_mad_csv(HF_DATA_URI)
        df.write_csv(DATA_PATH)
    except Exception as e:
        raise RuntimeError(
            'Не удалось прочитать hf://datasets/yandex/mad-cars/mad.csv. '
            'Проверь авторизацию huggingface-cli login или доступ к сети.'
        ) from e

print('HF dataset:', HF_DATA_URI)
print('Shape:', df.shape)
print('CSV path:', DATA_PATH.resolve())
print('CSV size (MB):', round(DATA_PATH.stat().st_size / (1024 * 1024), 2))

HF dataset: hf://datasets/yandex/mad-cars/mad.csv
Shape: (5884329, 6)
CSV path: /root/root/carcv/mad.csv
CSV size (MB): 531.96


In [8]:
check_creation(DATA_DIR)

'Каталог/файл root/carcv создан!'

In [25]:
URL_RE = re.compile(r'^https://storage\.yandexcloud\.net/yandex-research/mad/(\d+)/(\d+)\.jpg$')

In [10]:
brand_counter = Counter()
model_counter = Counter()
color_counter = Counter()
view_counter = Counter()
rows_per_car = Counter()
models_per_brand = defaultdict(set)
missing_counter = Counter()

total_rows = 0

with DATA_PATH.open('r', encoding='utf-8', newline='') as f:
    reader = csv.DictReader(f)
    columns = reader.fieldnames

    url_ok = 0
    url_bad_examples = []

    for row in reader:
        total_rows += 1

        brand = row['brand']
        model = row['model']
        color = row['color']
        car_id = row['car_id']
        view_id = row['view_id']
        url = row['url']

        if not brand:
            missing_counter['brand'] += 1
        if not model:
            missing_counter['model'] += 1
        if not color:
            missing_counter['color'] += 1

        brand_counter[brand] += 1
        model_counter[model] += 1
        color_counter[color] += 1
        view_counter[view_id] += 1
        rows_per_car[car_id] += 1

        if brand and model:
            models_per_brand[brand].add(model)

        m = URL_RE.match(url)
        if m and m.group(1) == car_id and m.group(2) == view_id:
            url_ok += 1
        elif len(url_bad_examples) < 10:
            url_bad_examples.append({'car_id': car_id, 'view_id': view_id, 'url': url})

rows_per_car_values = list(rows_per_car.values())

summary = {
    'shape': {'rows': total_rows, 'cols': len(columns), 'columns': columns},
    'uniques': {
        'brand': len(brand_counter),
        'model': len(model_counter),
        'color': len(color_counter),
        'car_id': len(rows_per_car),
        'view_id': len(view_counter),
    },
    'missing': dict(missing_counter),
    'rows_per_car': {
        'min': min(rows_per_car_values),
        'max': max(rows_per_car_values),
        'mean': round(mean(rows_per_car_values), 3),
        'median': median(rows_per_car_values),
    },
    'url_consistency': {
        'ok_ratio': round(url_ok / total_rows, 6),
        'bad_examples': url_bad_examples,
    },
}

summary

{'shape': {'rows': 5884329,
  'cols': 6,
  'columns': ['color', 'model', 'brand', 'car_id', 'view_id', 'url']},
 'uniques': {'brand': 143,
  'model': 1464,
  'color': 17,
  'car_id': 69971,
  'view_id': 100},
 'missing': {'brand': 588, 'model': 588, 'color': 378},
 'rows_per_car': {'min': 1, 'max': 200, 'mean': 84.097, 'median': 89},
 'url_consistency': {'ok_ratio': 1.0, 'bad_examples': []}}

In [11]:
def topn(counter: Counter, n: int = 20):
    return counter.most_common(n)


def bottomn(counter: Counter, n: int = 20):
    return sorted(counter.items(), key=lambda x: x[1])[:n]


def print_table(rows, title: str, left='key', right='count'):
    print(f'\n{title}')
    print('-' * len(title))
    print(f'{left:<32} {right:>12}')
    print(f"{'-' * 32} {'-' * 12}")
    for k, v in rows:
        label = str(k) if str(k) else '<EMPTY>'
        print(f'{label:<32} {v:>12}')

print_table(topn(brand_counter, 15), 'Top-15 brands', 'brand', 'rows')
print_table(topn(model_counter, 15), 'Top-15 models', 'model', 'rows')
print_table(topn(color_counter, 15), 'Top-15 colors (hex)', 'color', 'rows')

print_table(bottomn(brand_counter, 15), 'Bottom-15 brands', 'brand', 'rows')
print_table(bottomn(model_counter, 15), 'Bottom-15 models', 'model', 'rows')


Top-15 brands
-------------
brand                                    rows
-------------------------------- ------------
vaz                                    702785
kia                                    480231
toyota                                 398407
bmw                                    383548
hyundai                                373767
volkswagen                             357991
mercedes                               287793
nissan                                 257681
ford                                   236040
audi                                   204047
skoda                                  203937
renault                                186848
chevrolet                              169470
mazda                                  149827
mitsubishi                             143612

Top-15 models
-------------
model                                    rows
-------------------------------- ------------
rio                                    137867
focus                 

In [12]:
def quantile_from_sorted(values, q: float):
    idx = (len(values) - 1) * q
    lo = math.floor(idx)
    hi = math.ceil(idx)
    if lo == hi:
        return values[int(idx)]
    return values[lo] + (values[hi] - values[lo]) * (idx - lo)

brand_counts = sorted(brand_counter.values())
model_counts = sorted(model_counter.values())
color_counts = sorted(color_counter.values())
view_counts = sorted(view_counter.items(), key=lambda x: int(x[0]) if str(x[0]).isdigit() else 10**9)

long_tail_report = {
    'brand_q': {
        'q10': round(quantile_from_sorted(brand_counts, 0.10), 2),
        'q50': round(quantile_from_sorted(brand_counts, 0.50), 2),
        'q90': round(quantile_from_sorted(brand_counts, 0.90), 2),
    },
    'model_q': {
        'q10': round(quantile_from_sorted(model_counts, 0.10), 2),
        'q50': round(quantile_from_sorted(model_counts, 0.50), 2),
        'q90': round(quantile_from_sorted(model_counts, 0.90), 2),
    },
    'color_q': {
        'q10': round(quantile_from_sorted(color_counts, 0.10), 2),
        'q50': round(quantile_from_sorted(color_counts, 0.50), 2),
        'q90': round(quantile_from_sorted(color_counts, 0.90), 2),
    },
    'rare_classes': {
        'brands_lt_1k': sum(1 for v in brand_counter.values() if v < 1000),
        'models_lt_100': sum(1 for v in model_counter.values() if v < 100),
        'models_lt_1k': sum(1 for v in model_counter.values() if v < 1000),
    },
    'view_distribution_sample': view_counts[:20],
    'brands_with_max_models': sorted(
        ((b, len(models)) for b, models in models_per_brand.items()),
        key=lambda x: x[1],
        reverse=True,
    )[:15],
}

long_tail_report

{'brand_q': {'q10': 103.6, 'q50': 1872, 'q90': 142587.2},
 'model_q': {'q10': 89.0, 'q50': 539.0, 'q90': 9387.1},
 'color_q': {'q10': 20548.0, 'q50': 122763, 'q90': 1062523.0},
 'rare_classes': {'brands_lt_1k': 60,
  'models_lt_100': 245,
  'models_lt_1k': 907},
 'view_distribution_sample': [('0', 63202),
  ('1', 63604),
  ('2', 63394),
  ('3', 62743),
  ('4', 62257),
  ('5', 61458),
  ('6', 61088),
  ('7', 60801),
  ('8', 60315),
  ('9', 60042),
  ('10', 59670),
  ('11', 59774),
  ('12', 59557),
  ('13', 59405),
  ('14', 59409),
  ('15', 59432),
  ('16', 59169),
  ('17', 59223),
  ('18', 59044),
  ('19', 59024)],
 'brands_with_max_models': [('toyota', 129),
  ('nissan', 73),
  ('honda', 59),
  ('mercedes', 56),
  ('hyundai', 49),
  ('mitsubishi', 49),
  ('volkswagen', 46),
  ('audi', 45),
  ('ford', 42),
  ('mazda', 40),
  ('kia', 39),
  ('chevrolet', 38),
  ('bmw', 34),
  ('chery', 31),
  ('vaz', 29)]}

In [26]:
# Мини-диагностика для обучения VehicleMakeNet / VehicleTypeNet

train_readiness = {
    'has_missing_labels': any(summary['missing'].get(k, 0) > 0 for k in ('brand', 'model', 'color')),
    'url_consistent': summary['url_consistency']['ok_ratio'] == 1.0,
    'long_tail_severe_models': long_tail_report['rare_classes']['models_lt_100'] > 0,
    'long_tail_severe_brands': long_tail_report['rare_classes']['brands_lt_1k'] > 0,
}

train_readiness


{'has_missing_labels': True,
 'url_consistent': True,
 'long_tail_severe_models': True,
 'long_tail_severe_brands': True}

## Что делать дальше по изображениям

Если `progress_pct` < 100 (загрузка не завершена):
- повторно запустить ячейку download — `dst.exists()` пропустит уже скачанные файлы;
- проверить `missing` пары: возможно, URL недоступны (http_404).

Если `very_dark_ratio` или `very_blurry_ratio` высокие:
- добавить фильтрацию по quality-threshold перед обучением;
- либо оставить в train как hard-сэмплы и усилить аугментации.

Если есть `corrupted_files`:
- удалить битые файлы и перезапустить download — они будут перескачаны.

## Загрузка: ≤5 изображений на автомобиль

Стратегия: для каждого `car_id` берём до 5 `view_id`, равномерно распределённых по доступным ракурсам.  
Результат: `data/external/ymad_cars/images/{car_id}_{view_id}.jpg`  
Манифест: `data/external/ymad_cars/manifest.csv` (для resumability).

In [27]:
IMG_DIR = Path('root/carcv/data/external/ymad_cars/images')
IMG_DIR.mkdir(parents=True, exist_ok=True)

MANIFEST_PATH = Path('root/carcv/data/external/ymad_cars/manifest.csv')
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

In [28]:
print(check_creation(IMG_DIR))
check_creation(MANIFEST_PATH)

Каталог/файл root/carcv/data/external/ymad_cars/images создан!


'Каталог/файл root/carcv/data/external/ymad_cars/manifest.csv создан!'

In [29]:
MAX_IMGS_PER_CAR = 5

car_views: dict[str, list[int]] = defaultdict(list)
with DATA_PATH.open('r', encoding='utf-8', newline='') as f:
    for row in csv.DictReader(f):
        car_views[row['car_id']].append(int(row['view_id']))


def pick_evenly(views: list[int], k: int) -> list[int]:
    v = sorted(set(views))
    if len(v) <= k:
        return v
    return [v[round(i * (len(v) - 1) / (k - 1))] for i in range(k)]


manifest = []
for car_id, views in car_views.items():
    for vid in pick_evenly(views, MAX_IMGS_PER_CAR):
        manifest.append({
            'car_id': car_id,
            'view_id': vid,
            'fname': f'{car_id}_{vid}.jpg',
            'url': f'https://storage.yandexcloud.net/yandex-research/mad/{car_id}/{vid}.jpg',
        })

import csv as _csv
with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as f:
    w = _csv.DictWriter(f, fieldnames=['car_id', 'view_id', 'fname', 'url'])
    w.writeheader()
    w.writerows(manifest)

print(f'Cars: {len(car_views):,}  |  Images to fetch: {len(manifest):,}  |  Saved: {MANIFEST_PATH.resolve()}')

Cars: 69,971  |  Images to fetch: 349,795  |  Saved: /root/root/carcv/data/external/ymad_cars/manifest.csv


In [30]:
import resource
resource.setrlimit(resource.RLIMIT_NOFILE, (65536, 65536))

In [31]:
CONCURRENCY = 64
TIMEOUT_S = 30


async def _fetch(session: aiohttp.ClientSession, sem: asyncio.Semaphore, item: dict, out_dir: Path) -> str:
    dst = out_dir / item['fname']
    if dst.exists():
        return 'skip'
    async with sem:
        try:
            async with session.get(item['url'], timeout=aiohttp.ClientTimeout(total=TIMEOUT_S)) as r:
                if r.status == 200:
                    dst.write_bytes(await r.read())
                    return 'ok'
                return f'http_{r.status}'
        except Exception as e:
            return f'err_{type(e).__name__}'


async def download_all(items: list[dict], out_dir: Path) -> dict[str, int]:
    sem = asyncio.Semaphore(CONCURRENCY)
    stats: Counter = Counter()
    connector = aiohttp.TCPConnector(limit=CONCURRENCY)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [_fetch(session, sem, item, out_dir) for item in items]
        async for result in atqdm(asyncio.as_completed(tasks), total=len(tasks), desc='download'):
            stats[await result] += 1
    return dict(stats)


stats = await download_all(manifest, IMG_DIR)
print(stats)

download: 100%|██████████| 349795/349795 [39:40<00:00, 146.92it/s]  

{'skip': 119826, 'err_OSError': 229956, 'err_TimeoutError': 13}


In [32]:
_bytes = (
    sum(p.stat().st_size for p in IMG_DIR.rglob("*") if p.is_file())
    if IMG_DIR.exists()
    else 0
)
print(f"Размер каталога изображений: {_bytes / 1024**3:.2f} GiB ({_bytes:,} байт)")

Размер каталога изображений: 30.81 GiB (33,078,163,180 байт)


## EDA скачанных изображений (`data/external/ymad_cars`)

Анализируем изображения, скачанные на удалённый хост. Запускать **после** секции «Загрузка».

- Каталог: `data/external/ymad_cars/images`
- Манифест: `data/external/ymad_cars/manifest.csv`
- Формат имён: `{car_id}_{view_id}.jpg`

In [33]:
print('IMG_DIR =', IMG_DIR.resolve())
print('exists =', IMG_DIR.exists())
print('manifest exists =', MANIFEST_PATH.exists())

img_files = sorted(IMG_DIR.glob('*.jpg')) if IMG_DIR.exists() else []
print('jpg files:', len(img_files))
if img_files:
    print('first 10:', [p.name for p in img_files[:10]])

IMG_DIR = /root/root/carcv/data/external/ymad_cars/images
exists = True
manifest exists = True
jpg files: 119826
first 10: ['100002_2.jpg', '100002_28.jpg', '100002_50.jpg', '100002_72.jpg', '100002_99.jpg', '100004_0.jpg', '100004_22.jpg', '100004_47.jpg', '100004_72.jpg', '100004_99.jpg']


In [34]:
# Coverage: manifest vs фактически скачанные файлы на хосте

local_pairs: set[tuple[str, str]] = set()
bad_name_examples: list[str] = []

for p in img_files:
    parts = p.stem.split('_')
    if len(parts) != 2 or not parts[0].isdigit() or not parts[1].isdigit():
        if len(bad_name_examples) < 20:
            bad_name_examples.append(p.name)
        continue
    local_pairs.add((parts[0], parts[1]))

manifest_pairs: set[tuple[str, str]] = set()
if MANIFEST_PATH.exists():
    with MANIFEST_PATH.open('r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            manifest_pairs.add((row['car_id'], str(row['view_id'])))

downloaded = local_pairs & manifest_pairs
missing = manifest_pairs - local_pairs

coverage = {
    'manifest_total': len(manifest_pairs),
    'downloaded': len(downloaded),
    'missing': len(missing),
    'extra_local': len(local_pairs - manifest_pairs),
    'progress_pct': round(100 * len(downloaded) / max(1, len(manifest_pairs)), 2),
    'bad_name_examples': bad_name_examples,
}

coverage

{'manifest_total': 349795,
 'downloaded': 119826,
 'missing': 229969,
 'extra_local': 0,
 'progress_pct': 34.26,
 'bad_name_examples': []}

In [ ]:
# Quality scan: размеры, битые кадры, темнота, blur
# blur_metric = variance of Laplacian (чем меньше, тем более размыто)

size_counter = Counter()
brightness_values = []
blur_values = []
corrupted = []
very_dark = 0
very_blurry = 0

DARK_THR = 35.0
BLUR_THR = 40.0

for p in img_files:
    img = cv2.imread(str(p), cv2.IMREAD_COLOR)
    if img is None:
        if len(corrupted) < 20:
            corrupted.append(p.name)
        continue

    h, w = img.shape[:2]
    size_counter[(w, h)] += 1

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    bright = float(gray.mean())
    blur = float(cv2.Laplacian(gray, cv2.CV_64F).var())

    brightness_values.append(bright)
    blur_values.append(blur)

    if bright < DARK_THR:
        very_dark += 1
    if blur < BLUR_THR:
        very_blurry += 1

quality_report = {
    'total_files': len(img_files),
    'decoded_files': len(brightness_values),
    'corrupted_files': len(corrupted),
    'corrupted_examples': corrupted,
    'top_10_sizes': size_counter.most_common(10),
    'brightness': {
        'mean': round(float(np.mean(brightness_values)), 2) if brightness_values else None,
        'std': round(float(np.std(brightness_values)), 2) if brightness_values else None,
        'p05': round(float(np.percentile(brightness_values, 5)), 2) if brightness_values else None,
        'p50': round(float(np.percentile(brightness_values, 50)), 2) if brightness_values else None,
        'p95': round(float(np.percentile(brightness_values, 95)), 2) if brightness_values else None,
        'very_dark_count': very_dark,
        'very_dark_ratio': round(very_dark / max(1, len(brightness_values)), 4),
    },
    'blur': {
        'mean': round(float(np.mean(blur_values)), 2) if blur_values else None,
        'std': round(float(np.std(blur_values)), 2) if blur_values else None,
        'p05': round(float(np.percentile(blur_values, 5)), 2) if blur_values else None,
        'p50': round(float(np.percentile(blur_values, 50)), 2) if blur_values else None,
        'p95': round(float(np.percentile(blur_values, 95)), 2) if blur_values else None,
        'very_blurry_count': very_blurry,
        'very_blurry_ratio': round(very_blurry / max(1, len(blur_values)), 4),
    },
}

quality_report

In [ ]:
# Sanity-check: первые строки манифеста, которые уже скачаны

sample_rows: list[dict] = []
if MANIFEST_PATH.exists():
    with MANIFEST_PATH.open('r', encoding='utf-8', newline='') as f:
        for row in csv.DictReader(f):
            if (row['car_id'], str(row['view_id'])) in local_pairs:
                sample_rows.append(row)
            if len(sample_rows) >= 5:
                break

sample_rows, len(sample_rows)

## Выводы (по текущему прогону)

Ключевые наблюдения:
- Размер: **5,884,329** строк, **6** колонок.
- Кардинальности: `brand=143`, `model=1464`, `color=17`, `car_id=69,971`, `view_id=100`.
- Пропуски: `model=588`, `brand=588`, `color=378`.
- Сильный long-tail: `907` моделей имеют `<1000` примеров, из них `245` моделей имеют `<100`.
- `view_id` покрывается неравномерно (есть авто с 1 view, есть с 100 views).
- URL-структура валидна и согласована с `car_id/view_id` (ratio ~ 1.0).

Рекомендуемые действия перед train:
1. Удалить/исправить строки с пустыми `brand/model/color`.
2. Решить long-tail (merge редких классов, re-sampling, class-weighted loss, focal loss).
3. Делать split на уровне `car_id`, чтобы избежать leakage между train/val/test.
4. Проверить и при необходимости дедуплицировать near-duplicate кадры внутри одного `car_id` (особенно для dense view-sequences).
5. Для цвета рассмотреть маппинг hex → ограниченный набор color-классов (если downstream не использует raw hex).

---

Если нужно, следующим шагом соберу второй ноутбук `eda_images.ipynb`, который:
- скачает стратифицированный сэмпл изображений по `brand/model/color`;
- проверит реальное качество данных (битые файлы, размытость, темнота, дубликаты по перцептивному хэшу);
- даст визуальные галереи edge-cases для night/rain/blur.